# Módulo 11 de 12. DevOps y despliegue profesional

## Clase 5. Despliegue del frontend en Vercel y conexión con la API productiva

**Duración:** 180 minutos. **Material previo:** el directorio `frontend/` contiene una aplicación React completa para Universidad Digital; el backend FastAPI y PostgreSQL remoto se configuraron en la Clase 3. La clase comienza con ese código entregado. **Producto de la sesión:** una URL Vercel de producción que muestre las pantallas reales y permita iniciar sesión y consultar datos de Railway. No se reserva tiempo para desarrollar el frontend desde cero.

**Objetivos de aprendizaje.** Publicar un frontend Vite/React desde GitHub, distinguir variables que terminan en el navegador de secretos del backend, conectar un dominio HTTPS de Railway con el origen exacto de Vercel, comprender `vite.config.js`, `.env.production` y `vercel.json`, y explicar un error de build o de CORS con evidencia. La aplicación ya presenta roles de administrador, docente y estudiante; los controles del navegador reflejan los permisos que valida la API.

| Tiempo | Trabajo de clase | Evidencia |
|---|---|---|
| 0–20 min | Arquitectura completa y revisión del frontend entregado | Identifica interfaz, API y base de datos. |
| 20–45 min | Node.js, instalación local y build | `npm ci`, `npm run dev`, `npm run build`. |
| 45–70 min | Variables Vite y conexión HTTPS a Railway | URL correcta y explicación de su exposición. |
| 70–110 min | GitHub → Vercel con `frontend/` como raíz | Despliegue y dominio Vercel. |
| 110–145 min | CORS, login y navegación profunda | API responde desde Vercel. |
| 145–170 min | Diagnóstico y asistencia IA | Hipótesis comprobada en Network/Build Logs. |
| 170–180 min | Evidencias y evaluación | URL y registro de pruebas. |

Si la API remota de la Clase 3 todavía no responde en `/health/ready`, el grupo debe corregirla antes de afirmar que el sistema completo funciona. Los límites y condiciones de los planes GitHub, Railway y Vercel deben consultarse en los paneles vigentes de cada cuenta; no se presupone un crédito gratuito específico.

## 1. Tres piezas, tres responsabilidades

React dibuja pantallas y recoge acciones. FastAPI verifica identidad, aplica roles y decide si puede crear una inscripción o una nota. PostgreSQL conserva los registros; el navegador nunca se conecta directamente a esa base. En producción, Vercel sirve archivos estáticos compilados y Railway responde a solicitudes HTTPS. Un usuario que abre el dominio de Vercel descarga JavaScript que envía un `POST /api/v1/auth/token` a la URL pública de Railway; las lecturas posteriores añaden `Authorization: Bearer ...`. El backend consulta PostgreSQL mediante referencias privadas de Railway. Cambiar la visibilidad de un botón no cambia la autorización del servidor.

Observe el código entregado antes de instalar nada. `frontend/src/api/client.js` centraliza URL, JSON, token y errores; `src/auth/AuthContext.jsx` mantiene la sesión solo en memoria; `src/pages/` organiza usuarios, materias, períodos, inscripciones, notas y reportes; `vercel.json` permite recargar rutas como `/grades`. La persona estudiante puede revisar y ajustar ese código, pero su implementación no se cuenta como ejercicio de la clase. Los roles y rutas coinciden con el backend de Universidad Digital; consulte `frontend/README.md` para límites del MVP.

## 2. Instalar Node.js y comprobar el proyecto local paso a paso

En Windows abra PowerShell y ejecute `node --version` y `npm --version`. Vite 7 exige Node.js 20.19+ o 22.12+; elija una rama LTS compatible. Si falta Node, vaya a [nodejs.org/download](https://nodejs.org/en/download), seleccione el instalador Windows para la arquitectura del equipo, ejecútelo, mantenga la instalación de npm y la opción que incorpora Node al PATH, termine el asistente, cierre y abra PowerShell, y repita los dos comandos. Si la terminal continúa sin reconocer `node`, verifique la instalación y el PATH con la administración del equipo. Para macOS y Linux use las rutas de instalación de Node correspondientes al sistema. ([Vite: requisitos](https://vite.dev/guide/)).

Abra `universidad-digital/frontend` en la terminal de VS Code. `package-lock.json` ya fija el árbol de dependencias y `npm ci` instala exactamente esas versiones. En Windows:

```powershell
Copy-Item .env.example .env.local
npm ci
npm run dev
```

Abra <http://127.0.0.1:5173>. El archivo `.env.local` apunta a `http://127.0.0.1:8000` y se ignora en Git. Para esta comprobación la API local debe estar iniciada con Docker Compose o con Uvicorn, y `CORS_ORIGINS` del backend debe incluir exactamente `http://127.0.0.1:5173`. El ejemplo anterior del backend podía contener `http://localhost:5173`: `localhost` y `127.0.0.1` son orígenes distintos para el navegador. Agregue ambos si utiliza ambas direcciones y reinicie la API. En macOS/Linux el comando equivalente de copia es `cp .env.example .env.local`. Abra la página de login, sin necesidad de crear el frontend durante esta clase.

## 3. Entender `vite.config.js` y `.env.production`

En `vite.config.js`, `react()` transforma JSX; `server.port` configura el servidor local en 5173; `build.outDir` entrega los archivos de producción en `dist`. La función `loadEnv` exige que `VITE_API_BASE_URL` comience por `https://` en un build de producción. Así un despliegue sin URL pública falla antes de publicar una aplicación que apunte accidentalmente a `localhost`. El resto de la validación comprueba en `client.js` que la URL sea un origen sin ruta, usuario, contraseña ni consulta.

`VITE_API_BASE_URL` identifica **dónde** está la API y será visible en el código JavaScript enviado al navegador. Los nombres con prefijo `VITE_` se exponen al cliente en el build; no coloque allí `JWT_SECRET`, `ADMIN_PASSWORD`, `RAILWAY_TOKEN`, `DATABASE_URL` ni tokens de usuario. Para un build local de producción copie `.env.production.example` a `.env.production`, escriba únicamente `VITE_API_BASE_URL=https://SU-API.up.railway.app` con su dominio real y ejecute:

```powershell
Copy-Item .env.production.example .env.production
notepad .env.production
npm run build
npm run preview
```

Abra el URL que imprima `preview`. `.env.production` está en `.gitignore`: cada estudiante tiene su URL, mientras Vercel recibe el valor en su panel para el entorno **Production**. Una variable de build ya quedó incorporada al bundle; modificarla en el panel exige **un nuevo despliegue**. `npm run preview` prueba el bundle local, no equivale a comprobar el dominio final. ([Vite: variables y modos](https://vite.dev/guide/env-and-mode); [Vercel: variables de entorno](https://vercel.com/docs/environment-variables)).

## 4. Publicar en Vercel desde GitHub, pantalla por pantalla

Confirme en GitHub que el repositorio contiene `frontend/package.json`, `frontend/package-lock.json`, `frontend/vite.config.js` y `frontend/vercel.json`. Si el frontend se añadió después de la Clase 3, desde la raíz del repositorio ejecute `git status`, compruebe que no se incluirán `.env.local` o `.env.production`, y haga `git add frontend`, `git commit -m "Agregar frontend Universidad Digital"`, `git push origin main`. Si se trabaja mediante pull request, fusione solo después de las comprobaciones de su curso. El workflow del backend filtra cambios en `backend/`, así que un commit únicamente de frontend no inicia ese despliegue de Railway.

Abra [vercel.com](https://vercel.com/) e inicie sesión. Seleccione **Add New → Project → Import Git Repository**, conecte GitHub si Vercel solicita autorización y otorgue acceso al repositorio elegido. Seleccione `universidad-digital`. En la pantalla de configuración establezca **Root Directory: `frontend`** mediante **Edit**. Vercel debe reconocer Vite; confirme **Framework Preset: Vite**, **Install Command: `npm ci`**, **Build Command: `npm run build`** y **Output Directory: `dist`**. No use la raíz del monorepositorio, pues allí no existe el `package.json` del frontend. ([Vercel: repositorios Git](https://vercel.com/docs/git); [Vercel: Vite](https://vercel.com/docs/frameworks/frontend/vite)).

Antes de pulsar **Deploy**, despliegue la sección **Environment Variables** y cree `VITE_API_BASE_URL` con el origen HTTPS exacto de Railway, por ejemplo `https://universidad-api.up.railway.app`, asignado a **Production**. Si trabajará con vistas previas, configure también **Preview** con su backend de staging o, para un laboratorio controlado, con la API de práctica; cada dominio de Preview que haga peticiones directas debe estar autorizado en CORS. Pulse **Deploy**. Espere al estado **Ready**, inspeccione **Build Logs** si falla y copie el dominio de producción asignado. La conexión GitHub permite que los commits posteriores generen nuevos despliegues en Vercel; una vista previa por rama debe distinguirse de Production.

`vercel.json` contiene un rewrite de rutas a `/index.html`. Esta regla responde al problema de las SPA: al navegar dentro de React a `/subjects`, la aplicación funciona, pero una recarga directa pide esa ruta al servidor estático; el rewrite entrega el punto de entrada de React. No cambie su destino a Railway: la API se consulta explícitamente con `VITE_API_BASE_URL`. ([Vercel: rutas de Vite SPA](https://vercel.com/docs/frameworks/frontend/vite)).

## 5. Conectar el dominio real: CORS y la prueba de login

Copie el origen de Vercel, incluyendo `https://` y sin ruta ni barra final. En Railway abra **universidad-api → Variables**, busque `CORS_ORIGINS` y agregue el origen exacto, separado por coma de los orígenes locales que mantendrá: `http://127.0.0.1:5173,https://SU-FRONTEND.vercel.app`. Aplique los cambios y espere la nueva revisión saludable de Railway. No cambie las referencias privadas `DB_*`. CORS es una regla del navegador para autorizar el origen del frontend; no autentica personas ni reemplaza las comprobaciones de roles. `Authorization` provoca una solicitud previa `OPTIONS` que la configuración CORS de FastAPI debe permitir para ese origen.

Abra el sitio Vercel en una ventana privada. Verifique que aparece el login, autentíquese con un usuario académico de **prueba** y navegue a materias. Compare la pestaña **Network** del navegador: la petición de login debe dirigirse al dominio HTTPS de Railway y responder, y `GET /api/v1/users/me` debe devolver el perfil autorizado. Compruebe `/health/ready` en Railway por separado. Si el navegador bloquea CORS, copie el valor del encabezado `Origin` mostrado en Network y compárelo literalmente con la variable del backend; ajuste Railway y espere su nuevo despliegue. Si el error es 401, compruebe la cuenta y contraseña; si es 403 en una operación, compruebe rol y docente asignado. No publique contraseñas o encabezados `Authorization` en capturas.

## 6. IA como asistente de build y diagnóstico

Actividad guiada: entregue a la IA solo `package.json`, `vite.config.js`, el texto **sin credenciales** del error de build y esta petición: “Propón la causa más probable y una segunda hipótesis. Explica qué comando o pantalla de Vercel comprobaría cada una. Conserva `npm ci`, `dist`, Root Directory `frontend` y la regla de HTTPS; no añadas secretos a variables `VITE_`. Si sugieres cambiar un script, entrega el diff mínimo y una forma de probarlo”. El estudiante debe comparar esa respuesta con Build Logs y `npm run build`; aceptar un texto generado por IA sin ejecutarlo no constituye verificación.

Si la página carga y los datos no, inspeccione la secuencia: URL compilada → HTTPS → respuesta `/health/ready` → CORS → código HTTP → autorización → representación visual. Un error de carga que se limita a una ruta puede deberse al endpoint y no al build. Para estudiar una vista previa, documente su URL concreta y el backend al que se conecta; no use `CORS_ORIGINS=*` como atajo.

## 7. Cierre de la clase

La evidencia debe incluir URL de GitHub, URL productiva de Vercel, URL de salud de Railway, captura del build exitoso, captura de una consulta académica desde el frontend y explicación de 150–200 palabras: qué contiene `VITE_API_BASE_URL`, por qué no lleva secretos, para qué sirve el rewrite, dónde se configura CORS y qué exige repetir el build. El frontend desarrollado se entregó previamente; se evalúa la puesta en servicio y la conexión. La Clase 6 completará las pruebas de los tres roles y los escenarios de fallo.

**Criterios técnicos.** La configuración separada por entorno, el build reproducible, CORS explícito y las comprobaciones de disponibilidad contribuyen a seguridad, fiabilidad y mantenibilidad según ISO/IEC 25010:2023, sin afirmar que el MVP esté certificado o listo para datos institucionales. Consulte las fuentes oficiales enlazadas y la guía de `frontend/README.md` cuando cambien las interfaces de los proveedores.